# PIXELMIND – Sprint 3 : entraînement U-Net / Siamese U-Net (GPU)

Notebook prévu pour **Kaggle** (Colab fonctionne aussi). Il ne contient aucune logique : il
clone le dépôt et appelle les mêmes commandes qu'en local.

**Avant de lancer :**
- Kaggle → *Settings* → *Accelerator* : **GPU T4** ou **P100**. *Internet* : **On** (Kaggle
  demande une vérification par téléphone).
- Compter environ 1 à 1,5 h par modèle (100 epochs max, avec early stopping).

**Sortie :** `/kaggle/working/pixelmind_sprint3.zip` (checkpoints, base MLflow, résultats
test). Une fois téléchargé, il se décompresse à la racine du dépôt local (voir la dernière
cellule).

LEVIR-CD est réservé à un usage académique et non commercial.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/filou337/Pixelmind.git
%cd Pixelmind
!git log --oneline -1

In [ ]:
# torch is already installed on Kaggle/Colab with CUDA: it is not reinstalled
!pip install -q -e ".[ml,dl]"
import os

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

## 1. Données : téléchargement (MD5 vérifié) et contrôle d'identité avec le manifeste versionné

In [ ]:
!python -m pixelmind.data.download --root data/raw
!rm -rf data/raw/_archives   # saves 2.5 GB of disk

In [ ]:
# Rebuild the manifest from the downloaded files and compare checksums with the one in Git:
# guarantees we train on exactly the data analysed in Sprint 1.
!python -m pixelmind.data.manifest --root data/raw --out /tmp/levircd_check.jsonl
import json


def load(path):
    return {r["pair_id"]: r["sha256"] for r in map(json.loads, open(path))}


ref, now = load("data/manifests/levircd.jsonl"), load("/tmp/levircd_check.jsonl")
assert ref == now, "downloaded data differs from the versioned manifest"
print(f"OK: {len(ref)} pairs identical to the versioned manifest")

## 2. Entraînement

Configs : `configs/unet.yaml` et `configs/siamese_unet.yaml`. Pour surcharger un paramètre : `--set epochs=50 batch_size=8`.

In [ ]:
!python -m pixelmind.models.dl.train --config configs/unet.yaml

In [ ]:
!python -m pixelmind.models.dl.train --config configs/siamese_unet.yaml

## 3. Évaluation sur test (même protocole que les baselines : seuil choisi sur val)

In [ ]:
!python -m pixelmind.experiments.evaluate_checkpoint artifacts/models/unet_best.pt
!python -m pixelmind.experiments.evaluate_checkpoint artifacts/models/siamese_unet_diff_best.pt

## 4. Export

In [ ]:
!zip -qr /kaggle/working/pixelmind_sprint3.zip \
    artifacts/models artifacts/experiments/levircd
!ls -lh /kaggle/working/pixelmind_sprint3.zip

## En local, après téléchargement du zip

```bash
cd Pixelmind
unzip -o pixelmind_sprint3.zip        # -> artifacts/models, artifacts/experiments/levircd
python -m pixelmind.experiments.report   # updates docs/experiments/levircd_benchmark.md
mlflow ui --backend-store-uri sqlite:///artifacts/models/mlflow.db
```

Les JSON de résultats des baselines (Sprint 2) sont déjà dans `artifacts/experiments/levircd`
en local. Le zip n'apporte que les modèles DL, et le rapport fusionne le tout.